## Chuẩn bị môi trường


In [18]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

Sẵn sàng.


---
## Tải dữ liệu (đã cho)


In [2]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
## Loại bỏ cột rò rỉ nhãn (data leakage) và cột dư thừa

In [3]:
# TODO 1a: in tỷ lệ missing của tất cả các cột
print(df.isnull().mean())

# TODO 1b: bỏ các cột rò rỉ/dư thừa, gán lại vào biến df
leaky = ['alive', 'who','adult_male','class','deck','embark_town','alone'] # điền danh sách cột cần bỏ (chỉ những cột có trong df)
leaky = [col for col in leaky if col in df.columns]
df = df.drop(columns=leaky)

print("Các cột còn lại:", list(df.columns))

survived       0.000000
pclass         0.000000
sex            0.000000
age            0.198653
sibsp          0.000000
parch          0.000000
fare           0.000000
embarked       0.002245
class          0.000000
who            0.000000
adult_male     0.000000
deck           0.772166
embark_town    0.002245
alive          0.000000
alone          0.000000
dtype: float64
Các cột còn lại: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']


---
## Quan sát tổng quan

In [4]:
# TODO 2: shape, info, describe
# 1. In số dòng và số cột; biến mục tiêu (target)
print(f"Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")
print("Biến mục tiêu (target): 'survived'")

# 2. Xem kiểu dữ liệu và số giá trị non-null
print("-------------------------------")
print("Thông tin cấu trúc dữ liệu: ")
df.info()

# 3. Thống kế các biến số
print("-------------------------------")
print("Thống kê các biến số: ")
print(df.describe())

# 3. Thống kê các biến phân loại
print("-------------------------------")
print("Thống kê các biến phân loại: ")
print(df.describe(include=['object','category']))

Kích thước dữ liệu: 891 dòng, 8 cột
Biến mục tiêu (target): 'survived'
-------------------------------
Thông tin cấu trúc dữ liệu: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   sex       891 non-null    object 
 3   age       714 non-null    float64
 4   sibsp     891 non-null    int64  
 5   parch     891 non-null    int64  
 6   fare      891 non-null    float64
 7   embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB
-------------------------------
Thống kê các biến số: 
         survived      pclass         age       sibsp       parch        fare
count  891.000000  891.000000  714.000000  891.000000  891.000000  891.000000
mean     0.383838    2.308642   29.699118    0.523008    0.381594   32.204208
std      0.486592    0

---
## Missing Value: thống kê & đề xuất cách xử lý

In [5]:
# TODO 3: bảng missing (count + %)
missing_count = df.isnull().sum()
missing_percent = df.isnull().mean() * 100

missing_table = pd.DataFrame({
    'Số lượng thiếu: ': missing_count,
    'Phần trăm thiếu (%): ': missing_percent
})

print(missing_table[missing_table['Số lượng thiếu: '] > 0])

          Số lượng thiếu:   Phần trăm thiếu (%): 
age                    177              19.865320
embarked                 2               0.224467


---
## Phát hiện Outlier & **ra quyết định**


In [6]:
# TODO 4: đếm outlier theo IQR và Z-score cho 'age' và 'fare'
def dem_outlier_iqr(s):
    q1 = s.quantile(0.25)    # trả về số lượng outlier theo IQR
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return ((s < lower_bound) | (s > upper_bound)).sum()

def dem_outlier_zscore(s, nguong=3.0):
    s_clean = s.dropna()    # loại bỏ NaN trước khi tính toán z-score để tính toán lỗi
    z = np.abs(stats.zscore(s_clean))
    return (z > nguong).sum()  # trả về số lượng outlier theo Z-score

for col in ["age", "fare"]:
    print(f"Cột [{col}]:")
    print(f"  - Số outlier theo IQR: {dem_outlier_iqr(df[col])}")
    print(f"  - Số outlier theo Z-score: {dem_outlier_zscore(df[col])}")

Cột [age]:
  - Số outlier theo IQR: 11
  - Số outlier theo Z-score: 2
Cột [fare]:
  - Số outlier theo IQR: 116
  - Số outlier theo Z-score: 20


---
## Chia tập **TRƯỚC** khi tiền xử lý (chống data leakage)


In [11]:
# TODO 6: chia train/val/test có stratify
X = df.drop(columns=['survived'])
y = df['survived']

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.1765, random_state=42, stratify=y_tmp)

print(f"Train/Val/Test:{y_train.mean(): .4f} /{y_val.mean(): .4f} /{y_test.mean(): .4f}")
# in tỷ lệ survived từng tập

Train/Val/Test: 0.3836 / 0.3881 / 0.3806


---
## Xây pipeline tiền xử lý, **fit chỉ trên train**


In [12]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

# TODO 7: xây pipeline cho biến số và biến phân loại
pipe_so  = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)               # fit CHỈ trên train
X_train_t = preprocess.transform(X_train)
# ... transform cho val, test
X_val_t = preprocess.transform(X_val)
X_test_t = preprocess.transform(X_test)
print(X_train_t.shape, list(preprocess.get_feature_names_out()))

(623, 10) ['num__age', 'num__sibsp', 'num__parch', 'num__fare', 'cat__sex_female', 'cat__sex_male', 'cat__embarked_C', 'cat__embarked_Q', 'cat__embarked_S', 'ord__pclass']


---
## Training bằng Logistic Regression & Linear Regression

In [16]:
#Linear Regression
ln_model = LinearRegression()
ln_model.fit(X_train_t, y_train)

#Logistic Regression
log_model = LogisticRegression()
log_model.fit(X_train_t, y_train)

print("Huấn luyện thành công cả 2 mô hình!")

Huấn luyện thành công cả 2 mô hình!


---
## Dự đoán và đánh giá 2 mô hình

In [19]:

y_pred_ln_continuous = ln_model.predict(X_test_t)
y_pred_ln = np.where(y_pred_ln_continuous >= 0.5, 1, 0) # Áp dụng ngưỡng 0.5 để đưa về nhãn 0 (Không sống sót) hoặc 1 (Sống sót)

y_pred_log = log_model.predict(X_test_t)

def print_evaluation(y_true, y_pred, model_name):
    print(f"==== {model_name} ====")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("-" * 30 + "\n")

print_evaluation(y_test, y_pred_ln, "LINEAR REGRESSION")
print_evaluation(y_test, y_pred_log, "LOGISTIC REGRESSION")

==== LINEAR REGRESSION ====
Accuracy : 0.7612
Precision: 0.7021
Recall   : 0.6471
F1-score : 0.6735
Confusion Matrix:
[[69 14]
 [18 33]]
------------------------------

==== LOGISTIC REGRESSION ====
Accuracy : 0.7761
Precision: 0.7442
Recall   : 0.6275
F1-score : 0.6809
Confusion Matrix:
[[72 11]
 [19 32]]
------------------------------



---
## Nhận xét và So sánh
Dựa trên kết quả đánh giá (Accuracy, Precision, Recall, F1-score và Confusion Matrix), ta thấy:

1. **Hiệu năng:** Mô hình **Logistic Regression** cho kết quả đánh giá (đặc biệt là Accuracy và F1-score) ổn định và tốt hơn một ít so với Linear Regression sau khi áp ngưỡng.
2. **Bản chất thuật toán:**
   - **Linear Regression** sinh ra để giải quyết bài toán hồi quy (dự đoán một giá trị liên tục). Đầu ra của nó là một đường thẳng và các giá trị dự đoán có thể nằm ngoài khoảng [0, 1], vô lý đối với bài toán xác suất hoặc phân loại nhãn. Việc chọn ngưỡng 0.5 mang tính chất khiên cưỡng.
   - **Logistic Regression** sử dụng hàm Sigmoid để ép giá trị dự đoán luôn nằm trong khoảng [0, 1], biểu diễn chính xác xác suất thuộc về một lớp (sống sót hay không). Nó sinh ra đặc biệt để tối ưu cho bài toán phân loại.

**Kết luận:** Đối với bài toán phân loại hành khách sống sót (phân loại nhị phân), mô hình **Logistic Regression** là mô hình phù hợp, tự nhiên và đáng tin cậy hơn so với Linear Regression.